# Week 2: Statistical Models for Time Series Forecasting

- **Training:** 2013-01-01 to 2013-12-31
- **Test:** 2014-01-01 to 2014-03-31

In [ ]:
# Import libraries for modelling, evaluation, and plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load Data & Train/Test Split

In [ ]:
# Load cleaned data and split into train/test periods
df = pd.read_csv("data/cleaned_timeseries.csv", parse_dates=["date"])
df = df.sort_values("date").set_index("date").asfreq("D")

train = df.loc["2013-01-01":"2013-12-31"]
test = df.loc["2014-01-01":"2014-03-31"]

plt.plot(train.index, train["unit_sales"], label="Train")
plt.plot(test.index, test["unit_sales"], label="Test", color="orange")
plt.title("Train / Test Split")
plt.legend()
plt.show()
print(f"Train: {len(train)} days, Test: {len(test)} days")

## 2. Evaluation Helpers

In [ ]:
# Utility functions to evaluate and plot forecasts
def evaluate(true, pred, name):
    mse = mean_squared_error(true, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(true, pred)
    r2 = r2_score(true, pred)
    print(f"{name:15s}  MSE: {mse:.1f}  RMSE: {rmse:.1f}  MAE: {mae:.1f}  R²: {r2:.4f}")
    return {"Model": name, "MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

results = []

def plot_forecast(train, test, pred, title):
    plt.plot(train.index, train["unit_sales"], label="Train")
    plt.plot(test.index, test["unit_sales"], label="Actual", color="orange")
    plt.plot(test.index, pred, label="Forecast", color="red", linestyle="--")
    plt.title(title)
    plt.legend()
    plt.show()

## 3. Model 1: ARIMA

In [ ]:
# ADF test to determine differencing order, then grid-search ARIMA(p,1,q)
adf = adfuller(train["unit_sales"].dropna())
print(f"ADF p-value: {adf[1]:.4f} -> d=1")

best_aic, best_order, best_model = np.inf, None, None
for p in range(4):
    for q in range(4):
        try:
            m = ARIMA(train["unit_sales"], order=(p, 1, q)).fit()
            if m.aic < best_aic:
                best_aic, best_order, best_model = m.aic, (p, 1, q), m
        except Exception:
            pass

print(f"Best order: {best_order} (AIC: {best_aic:.1f})")
pred = best_model.forecast(len(test))
plot_forecast(train, test, pred, "ARIMA")
results.append(evaluate(test["unit_sales"], pred, "ARIMA"))

## 4. Model 2: SARIMA

In [ ]:
# SARIMA adds weekly seasonality to the best ARIMA order
order = best_order if best_order else (2, 1, 2)
m = SARIMAX(train["unit_sales"], order=order, seasonal_order=(1, 1, 1, 7),
            enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(m.summary())

pred = m.get_forecast(len(test)).predicted_mean
plot_forecast(train, test, pred, "SARIMA")
results.append(evaluate(test["unit_sales"], pred, "SARIMA"))

## 5. Model 3: Holt-Winters (Exponential Smoothing)

In [ ]:
# Triple exponential smoothing with additive trend and weekly seasonality
m = ExponentialSmoothing(train["unit_sales"], trend="add",
                         seasonal="add", seasonal_periods=7).fit()

pred = m.forecast(len(test))
plot_forecast(train, test, pred, "Holt-Winters")
results.append(evaluate(test["unit_sales"], pred, "Holt-Winters"))

## 6. Model Comparison

**Best model: Holt-Winters** (RMSE: 118, R²: 0.48). It captures the weekly seasonality naturally and handles the additive trend better than SARIMA (which had convergence warnings with this data). ARIMA without seasonality performs worst (R²: 0.14).

In [ ]:
# Rank models by RMSE and plot comparison
rdf = pd.DataFrame(results).sort_values("RMSE")
print(rdf.to_string(index=False))

plt.bar(rdf["Model"], rdf["RMSE"], color="#3498db", edgecolor="black")
plt.title("Model Comparison (RMSE)")
plt.ylabel("RMSE")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()